# 05 — Evaluation and Error Analysis

This notebook loads the tuned models, evaluates the untouched grouped test set, compares the median baseline with both model families, creates report figures, examines Ridge coefficients, and performs detailed error analysis.

### Imports — why this cell is needed
Loads the final evaluation, plotting, and model-loading libraries.

In [ ]:
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)

import joblib

### Repository paths — why this cell is needed
Loads engineered data and fitted models and saves final figures using only relative paths.

In [ ]:
def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the cloned "
        "CSE437 repository, with data/ and notebooks/ folders present."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)

### Load engineered data — why this cell is needed
Recreates the exact final test split from the engineered dataset.

In [ ]:
ENGINEERED_CSV = PROCESSED_DIR / "seattle_energy_features.csv"
if not ENGINEERED_CSV.exists():
    raise FileNotFoundError(
        "Run 03_feature_engineering.ipynb first. "
        "Expected data/processed/seattle_energy_features.csv"
    )

df = pd.read_csv(ENGINEERED_CSV, low_memory=False)
print("Loaded:", ENGINEERED_CSV.relative_to(ROOT))
print("Shape:", df.shape)

### Recreate the grouped test split — why this cell is needed
Uses the same random seed and building grouping as modelling so the untouched test set is identical.

In [ ]:
TARGET = "SiteEUI"
GROUP = "OSEBuildingID"

X = df.drop(columns=[TARGET, GROUP])
y = df[TARGET]
groups = df[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

dev_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_dev = X.iloc[dev_idx].reset_index(drop=True)
y_dev = y.iloc[dev_idx].reset_index(drop=True)
groups_dev = groups.iloc[dev_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

shared_buildings = set(groups_dev).intersection(set(groups_test))

print("Development rows:", len(X_dev))
print("Test rows:", len(X_test))
print("Development buildings:", groups_dev.nunique())
print("Test buildings:", groups_test.nunique())
print("Shared building IDs:", len(shared_buildings))

assert len(shared_buildings) == 0

### Define the custom clipping transformer — why this cell is needed
The saved pipelines contain this custom transformer, so it must be defined before loading the fitted models.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = X.copy()
        self.bounds_ = {}
        for col in self.columns or []:
            series = pd.to_numeric(X[col], errors="coerce")
            self.bounds_[col] = (
                series.quantile(self.lower),
                series.quantile(self.upper)
            )
        return self

    def transform(self, X):
        X = X.copy()
        for col, (low, high) in self.bounds_.items():
            if col in X.columns:
                X[col] = pd.to_numeric(
                    X[col], errors="coerce"
                ).clip(low, high)
        return X

### Match categorical dtypes — why this cell is needed
Uses standard Python objects and `np.nan`, matching the Colab-compatible representation used during model fitting.

In [ ]:
numeric_features = X_dev.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_dev.columns if c not in numeric_features]

for col in categorical_features:
    X_dev[col] = X_dev[col].astype(object)
    X_test[col] = X_test[col].astype(object)
    X_dev[col] = X_dev[col].where(pd.notna(X_dev[col]), np.nan)
    X_test[col] = X_test[col].where(pd.notna(X_test[col]), np.nan)

### Load tuned models — why this cell is needed
Loads the estimators produced by `04_modeling_and_tuning.ipynb` without re-running the searches.

In [ ]:
MODEL_BUNDLE = MODELS_DIR / "model_bundle.joblib"
if not MODEL_BUNDLE.exists():
    raise FileNotFoundError(
        "Run 04_modeling_and_tuning.ipynb first. "
        "Expected models/model_bundle.joblib"
    )

bundle = joblib.load(MODEL_BUNDLE)

ridge_best = bundle["ridge_model"]
boost_best = bundle["boost_model"]
ridge_cv_mae = bundle["ridge_cv_mae"]
boost_cv_mae = bundle["boost_cv_mae"]
tuned_comparison = bundle["tuned_comparison"]

print("Ridge validation MAE:", round(ridge_cv_mae, 3))
print("Gradient Boosting validation MAE:", round(boost_cv_mae, 3))

## Final test evaluation

### Select the final model and calculate final metrics — why this cell is needed
Selects by validation MAE only and evaluates once on unseen test buildings.

In [ ]:
if ridge_cv_mae <= boost_cv_mae:
    final_model_name = "Tuned Ridge Regression"
    final_model = ridge_best
else:
    final_model_name = "Tuned Histogram Gradient Boosting Regression"
    final_model = boost_best

test_predictions = np.maximum(final_model.predict(X_test), 0)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5
test_r2 = r2_score(y_test, test_predictions)
test_rmsle = mean_squared_log_error(y_test, test_predictions) ** 0.5
test_log_r2 = r2_score(np.log1p(y_test), np.log1p(test_predictions))

final_metrics = pd.DataFrame([{
    "Final Model": final_model_name,
    "Test MAE": test_mae,
    "Test RMSE": test_rmse,
    "Test R2": test_r2,
    "Test RMSLE": test_rmsle,
    "Test Log-Scale R2": test_log_r2
}])

display(final_metrics.round(3))

### Baseline and both-model test comparison — why this cell is needed
Provides the exact baseline/Ridge/Gradient Boosting comparison required by the report template.

In [ ]:
baseline_value = y_dev.median()
baseline_predictions = np.full(len(y_test), baseline_value, dtype=float)

ridge_test_predictions = np.maximum(ridge_best.predict(X_test), 0)
boost_test_predictions = np.maximum(boost_best.predict(X_test), 0)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "RMSLE": mean_squared_log_error(y_true, y_pred) ** 0.5,
        "Log-Scale R2": r2_score(np.log1p(y_true), np.log1p(y_pred))
    }

test_model_comparison = pd.DataFrame([
    {"Model": "Median Baseline", **regression_metrics(y_test, baseline_predictions)},
    {"Model": "Tuned Ridge Regression", **regression_metrics(y_test, ridge_test_predictions)},
    {"Model": "Tuned Histogram Gradient Boosting", **regression_metrics(y_test, boost_test_predictions)}
])

print(f"Development-set median used for baseline: {baseline_value:.3f}")
display(test_model_comparison.round(3))

### Final overfitting/underfitting diagnosis — why this cell is needed
Uses saved grouped-CV train and validation scores for the selected model.

In [ ]:
selected_label = (
    "Tuned Ridge"
    if final_model_name == "Tuned Ridge Regression"
    else "Tuned Gradient Boosting"
)

selected_tuned_row = tuned_comparison.loc[
    tuned_comparison["Model"].eq(selected_label)
].iloc[0]

train_mae = selected_tuned_row["Train MAE"]
val_mae = selected_tuned_row["Validation MAE"]
train_r2 = selected_tuned_row["Train R2"]
val_r2 = selected_tuned_row["Validation R2"]
r2_gap = train_r2 - val_r2

if train_mae < 0.75 * val_mae and r2_gap > 0.15:
    final_diagnosis = (
        "The model shows evidence of overfitting: training performance is "
        "substantially better than grouped-validation performance."
    )
elif train_r2 < 0.30 and val_r2 < 0.30:
    final_diagnosis = (
        "The model is not mainly overfitting. Both training and validation "
        "R² are low, indicating underfitting or limited predictive signal."
    )
else:
    final_diagnosis = (
        "There is no strong overfitting signal. Training and grouped-validation "
        "performance are reasonably close."
    )

print("Selected model:", final_model_name)
print("Grouped-CV Train MAE:", round(train_mae, 3))
print("Grouped-CV Validation MAE:", round(val_mae, 3))
print("Grouped-CV Train R²:", round(train_r2, 3))
print("Grouped-CV Validation R²:", round(val_r2, 3))
print("R² gap:", round(r2_gap, 3))
print("\nDiagnosis:")
print(final_diagnosis)

## Visualizations

### Actual vs predicted — why this cell is needed
Shows prediction performance on the original SiteEUI scale.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_predictions, alpha=0.35)
low = min(y_test.min(), test_predictions.min())
high = max(y_test.max(), test_predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual SiteEUI")
plt.ylabel("Predicted SiteEUI")
plt.title("Actual vs Predicted SiteEUI")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "actual_vs_predicted_siteeui.png", dpi=200, bbox_inches="tight")
plt.show()

### Log-scale actual vs predicted — why this cell is needed
Makes proportional performance across the typical SiteEUI range easier to see.

In [ ]:
actual_log = np.log1p(y_test)
predicted_log = np.log1p(test_predictions)

plt.figure(figsize=(6, 6))
plt.scatter(actual_log, predicted_log, alpha=0.35)
low_log = min(actual_log.min(), predicted_log.min())
high_log = max(actual_log.max(), predicted_log.max())
plt.plot([low_log, high_log], [low_log, high_log], linestyle="--")
plt.xlabel("Actual log1p(SiteEUI)")
plt.ylabel("Predicted log1p(SiteEUI)")
plt.title("Actual vs Predicted SiteEUI — Log Scale")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "actual_vs_predicted_log.png", dpi=200, bbox_inches="tight")
plt.show()

### Residual plot — why this cell is needed
Shows where prediction errors increase or become systematic.

In [ ]:
residuals = y_test.to_numpy() - test_predictions

plt.figure(figsize=(7, 4))
plt.scatter(test_predictions, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted SiteEUI")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "residual_plot.png", dpi=200, bbox_inches="tight")
plt.show()

### Ridge coefficient analysis — why this cell is needed
Provides the coefficient-based interpretation requested by the report template.

In [ ]:
ridge_pipeline = ridge_best
ridge_preprocessor = ridge_pipeline.named_steps["preprocess"]
ridge_feature_names = ridge_preprocessor.get_feature_names_out()

if "select" in ridge_pipeline.named_steps:
    selected_mask = ridge_pipeline.named_steps["select"].get_support()
    ridge_coefficient_names = ridge_feature_names[selected_mask]
elif "svd" in ridge_pipeline.named_steps:
    n_components = ridge_pipeline.named_steps["svd"].n_components
    ridge_coefficient_names = np.array(
        [f"SVD Component {i+1}" for i in range(n_components)]
    )
else:
    ridge_coefficient_names = ridge_feature_names

ridge_ttr = ridge_pipeline.named_steps["model"]
ridge_coefficients = np.asarray(ridge_ttr.regressor_.coef_).ravel()

ridge_coefficients_df = pd.DataFrame({
    "Feature": ridge_coefficient_names,
    "Coefficient": ridge_coefficients
})
ridge_coefficients_df["Absolute Coefficient"] = ridge_coefficients_df["Coefficient"].abs()

top_ridge_coefficients = (
    ridge_coefficients_df
    .sort_values("Absolute Coefficient", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

display(top_ridge_coefficients.round(4))

plot_coefficients = top_ridge_coefficients.sort_values("Coefficient")
plt.figure(figsize=(9, 6))
plt.barh(plot_coefficients["Feature"].astype(str), plot_coefficients["Coefficient"])
plt.xlabel("Ridge Coefficient")
plt.title("Top 15 Ridge Coefficients by Absolute Magnitude")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ridge_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()

## Error analysis

### Largest individual errors — why this cell is needed
Provides concrete examples of failure cases.

In [ ]:
error_df = X_test.copy()
error_df["ActualSiteEUI"] = y_test.values
error_df["PredictedSiteEUI"] = test_predictions
error_df["AbsoluteError"] = np.abs(
    error_df["ActualSiteEUI"]
    - error_df["PredictedSiteEUI"]
)
error_df["SquaredError"] = (
    error_df["ActualSiteEUI"]
    - error_df["PredictedSiteEUI"]
) ** 2

display_columns = [
    c for c in [
        "EPAPropertyType",
        "BuildingType",
        "BuildingAge",
        "PropertyGFATotal",
        "ActualSiteEUI",
        "PredictedSiteEUI",
        "AbsoluteError"
    ] if c in error_df.columns
]

print("10 largest test prediction errors:")
display(
    error_df
    .sort_values(
        "AbsoluteError",
        ascending=False
    )
    [display_columns]
    .head(10)
)

### Error by EPA property type — why this cell is needed
Identifies property subgroups where the final model performs worst.

In [ ]:
error_group_column = (
    "EPAPropertyType"
    if "EPAPropertyType" in error_df.columns
    else "BuildingType"
)

error_by_type = (
    error_df
    .groupby(
        error_group_column,
        dropna=False
    )
    .agg(
        Count=("AbsoluteError", "size"),
        MAE=("AbsoluteError", "mean"),
        RMSE=(
            "SquaredError",
            lambda x: np.sqrt(x.mean())
        )
    )
)

error_by_type = (
    error_by_type[
        error_by_type["Count"] >= 20
    ]
    .sort_values(
        "MAE",
        ascending=False
    )
)

display(error_by_type)

if len(error_by_type):
    plot_data = (
        error_by_type
        .head(10)
        .sort_values("MAE")
    )

    plt.figure(figsize=(8, 5))
    plt.barh(
        plot_data.index.astype(str),
        plot_data["MAE"]
    )
    plt.xlabel("Mean Absolute Error")
    plt.title(
        f"Highest Prediction Error by "
        f"{error_group_column}"
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "error_by_property_type.png", dpi=200, bbox_inches="tight")
    plt.show()

### Final summary — why this cell is needed
Collects the headline metrics, generalization diagnosis, and main failure categories.

In [ ]:
print("FINAL MODEL SUMMARY")
print("-" * 70)
print("Model:", final_model_name)
print(f"Test MAE: {test_mae:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Test R²: {test_r2:.3f}")
print(f"Test RMSLE: {test_rmsle:.3f}")
print(f"Log-scale R²: {test_log_r2:.3f}")
print("\nGeneralization diagnosis:")
print(final_diagnosis)

if len(error_by_type):
    print(f"\nHighest-error {error_group_column} categories with at least 20 test records:")
    for category, row in error_by_type.head(3).iterrows():
        print(
            f" - {category}: MAE={row['MAE']:.2f}, "
            f"RMSE={row['RMSE']:.2f}, n={int(row['Count'])}"
        )

print(
    "\nWhy errors remain: structural and property-use features cannot fully "
    "describe occupancy, operating hours, specialized equipment, laboratory/"
    "medical loads, or unusual operational behaviour. Direct energy-use, "
    "SourceEUI, emissions, and ENERGY STAR variables were intentionally "
    "excluded because they would cause target leakage."
)